In [6]:
!pip3 install faiss-cpu numpy scikit-learn "tensorflow==2.20.0" pyarrow==22.0 tensorflow-hub --break-system-packages

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.2/34.2 MB 103.3 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 21.0.0
    Uninstalling pyarrow-21.0.0:
      Successfully uninstalled pyarrow-21.0.0


### LSH (Locality-Sensitive Hashing)

The goal is to make similar vectors collide in the same hash buckets.

1. Step 1: apply random projections/hyperplanes to each vector.
2. Step 2: convert projection signs/values into a compact hash code.
3. Step 3: store vectors by hash buckets (or binary codes).

Query: hash the query the same way, then search matching/nearby buckets.

Result: very fast candidate filtering, then optional exact distance rerank.

Strengths: simple, fast indexing, scalable.

Tradeoff: approximate quality can drop for complex/high-precision search.

### HNSW (Hierarchical Navigable Small World)

Goal: build a graph where nearby vectors are connected for fast navigation.

Structure: multiple layers; top is sparse (long-range links), bottom is dense (local links).

Build step: insert each vector by descending layers and linking to nearest neighbors.

1. Query step 1: start at top entry point, greedily move to closer nodes.
2. Query step 2: descend layers, refining neighborhood.
3. Query step 3: at base layer, perform wider best-first exploration to get top-k neighbors.

Strengths: excellent recall-latency balance, strong practical ANN performance.

Tradeoff: higher memory usage and more index-tuning complexity (M, efConstruction, efSearch).

### Understanding Semantic Search

Semantic search improves on keyword search by understanding intent and context, not just exact words. It returns results that are more relevant and can improve over time based on user behavior.

##### How It Works

Using NLP, semantic search:
- Interprets the meaning of a query
- Understands relationships between terms (for example, synonyms)
- Ranks and returns the most relevant results

##### Technical Basics: Vectors

Semantic search represents text as vectors (numeric embeddings). Similar meanings produce vectors that are close together.

- **Vectorization**: Convert queries and documents into vectors (for example, with Universal Sentence Encoder)
- **Similarity**: Compare vectors using metrics like cosine similarity
- **Retrieval**: Return documents whose vectors are closest to the query vector

In short, semantic search retrieves results by meaning, not just matching words.

### Understanding Vectorization and Indexing

Vectorization and indexing power semantic search.  
- **Universal Sentence Encoder (USE)** converts sentences into vectors (numeric embeddings) that capture meaning and context.  
- **FAISS** indexes those vectors for fast similarity search at scale.

In brief,

1. USE uses deep learning to encode sentence meaning by analyzing words, context, and order, then outputs a vector for each sentence.

2. FAISS builds an index over vectors so similar vectors are grouped efficiently. For a query vector, it quickly finds nearest neighbors and returns the most relevant results.

3. USE provides meaningful vector representations; FAISS makes retrieval fast and scalable. Combined, they enable accurate, efficient semantic search.

In [7]:
import numpy as np
import tensorflow as tf
import tensorflow_hub as hub
import faiss
import re
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from pprint import pprint

# Suppressing warnings
def warn(*args, **kwargs):
    pass

import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

### The 20 Newsgroups Dataset

The 20 Newsgroups dataset contains about 20,000 documents across 20 topics (for example, sports, science, politics, religion). It’s widely used in NLP because it includes real discussions with varied language and strong context.

##### How we’ll use it
1. Load and inspect the dataset.
2. Clean and preprocess the text.
3. Convert documents to vectors with Universal Sentence Encoder.
4. Index vectors with FAISS for fast semantic search.

This gives hands-on practice building a full semantic search pipeline on realistic data.

In [8]:
from sklearn.datasets import fetch_20newsgroups
newsgroups_train = fetch_20newsgroups(subset='train')

### Pre-processing Dataset

In [10]:
newsgroups = fetch_20newsgroups(subset='all')
documents = newsgroups.data

# Basic preprocessing of text data
def preprocess_text(text):
    # Remove email headers
    text = re.sub(r'^From:.*\n?', '', text, flags=re.MULTILINE)
    # Remove email addresses
    text = re.sub(r'\S*@\S*\s?', '', text)
    # Remove punctuations and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Convert to lowercase
    text = text.lower()
    # Remove excess whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Preprocess each document
processed_documents = [preprocess_text(doc) for doc in documents]
# Choose a sample post to display
sample_index = 0  # for example, the first post in the dataset

# Print the original post
print("Original post:\n")
print(newsgroups_train.data[sample_index])
print("\n" + "-"*80 + "\n")

# Print the preprocessed post
print("Preprocessed post:\n")
print(preprocess_text(newsgroups_train.data[sample_index]))
print("\n" + "-"*80 + "\n")

Original post:

From: lerxst@wam.umd.edu (where's my thing)
Subject: WHAT car is this!?
Nntp-Posting-Host: rac3.wam.umd.edu
Organization: University of Maryland, College Park
Lines: 15

 I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.

Thanks,
- IL
   ---- brought to you by your neighborhood Lerxst ----






--------------------------------------------------------------------------------

Preprocessed post:

subject what car is this nntppostinghost racwamumdedu organization university of maryland college park lines i was wondering if anyone out there could enlighte

### Universal Sentence Encoder

After preprocessing the text data, the next step is to transform this cleaned text into numerical vectors using the Universal Sentence Encoder (USE). These vectors capture the semantic essence of the text.

By vectorizing the text with USE, we've now converted our textual data into a format that can be efficiently processed by machine learning algorithms, setting the stage for the next step: indexing with FAISS.

In [11]:
# Load the Universal Sentence Encoder's TF Hub module
embed = hub.load("https://tfhub.dev/google/universal-sentence-encoder/4")

# Function to generate embeddings
def embed_text(text):
    return embed(text).numpy()

# Generate embeddings for each preprocessed document
X_use = np.vstack([embed_text([doc]) for doc in processed_documents])

### Indexing with FAISS

<img src="https://ik.imagekit.io/upgrad1/abroad-images/imageCompo/images/ChatGPT_Image_Feb_9_2026_11_16_23_AM8KR5UZ.png" width="350px">

In practice:

1. Text/data points are converted to embeddings (numeric vectors).

2. A distance metric is chosen (usually cosine or Euclidean).

3. The clustering algorithm groups points that are close under that metric:

    - k-means: picks k centers and assigns each point to nearest center.
    - DBSCAN/HDBSCAN: finds dense regions and labels sparse points as noise.
    - Hierarchical: repeatedly merges nearest groups.

4. Then a search would point to the most related result.

With our documents now represented as vectors using the Universal Sentence Encoder, the next step is to use FAISS (Facebook AI Similarity Search) for efficient similarity searching.

FAISS offers a variety of indexes tailored for different use cases and dataset sizes. Depending on your specific needs and the complexity of your data, you might consider other indexes for more efficient searching. In this project, we use `IndexFlatL2` for its simplicity and effectiveness in handling small to medium-sized datasets.

For larger datasets or more advanced use cases, indexes like `IndexIVFFlat`, `IndexIVFPQ`, and others can provide faster search times and reduced memory usage. Explore more at [FAISS indexes wiki](https://github.com/facebookresearch/faiss/wiki/Faiss-indexes).

In [12]:
dimension = X_use.shape[1]
index = faiss.IndexFlatL2(dimension)  # Creating a FAISS index
index.add(X_use)  # Adding the document vectors to the index

### Querying with FAISS

The `search` function finds semantically similar documents for a query by:
- Preprocessing the query
- Converting it to an embedding
- Using FAISS to retrieve the top `k` nearest neighbors
- Returning neighbor distances and indices

When running a sample query (for example, `"motorcycle"`), results show:
- Rank
- Distance (similarity signal)
- Matched document text (preprocessed and original)

This demonstrates semantic search: retrieving contextually relevant results, not just keyword matches.

In [13]:
# Function to perform a query using the Faiss index
def search(query_text, k=5):
    # Preprocess the query text
    preprocessed_query = preprocess_text(query_text)
    # Generate the query vector
    query_vector = embed_text([preprocessed_query])
    # Perform the search
    distances, indices = index.search(query_vector.astype('float32'), k)
    return distances, indices

# Example Query
query_text = "motorcycle"
distances, indices = search(query_text)

# Display the results
for i, idx in enumerate(indices[0]):
    # Ensure that the displayed document is the preprocessed one
    print(f"Rank {i+1}: (Distance: {distances[0][i]})\n{processed_documents[idx]}\n")

Rank 1: (Distance: 1.0367169380187988)
subject first bike organization freshman mechanical engineering carnegie mellon pittsburgh pa lines nntppostinghost andrewcmuedu anyone i am a serious motorcycle enthusiast without a motorcycle and to put it bluntly it sucks i really would like some advice on what would be a good starter bike for me i do know one thing however i need to make my first bike a good one because buying a second any time soon is out of the question i am specifically interested in racing bikes cbr f gsxr i know that this may sound kind of crazy considering that ive never had a bike before but i am responsible a fast learner and in love please give me any advice that you think would help me in my search including places to look or even specific bikes that you want to sell me thanks jamie belliveau

Rank 2: (Distance: 1.0392436981201172)
subject first bike organization freshman mechanical engineering carnegie mellon pittsburgh pa lines nntppostinghost poandrewcmuedu anyone

In [14]:
# Display the results
for i, idx in enumerate(indices[0]):
    # Displaying the original (unprocessed) document corresponding to the search result
    print(f"Rank {i+1}: (Distance: {distances[0][i]})\n{documents[idx]}\n")

Rank 1: (Distance: 1.0367169380187988)
From: James Leo Belliveau <jbc9+@andrew.cmu.edu>
Subject: First Bike??
Organization: Freshman, Mechanical Engineering, Carnegie Mellon, Pittsburgh, PA
Lines: 17
NNTP-Posting-Host: andrew.cmu.edu

 Anyone, 

    I am a serious motorcycle enthusiast without a motorcycle, and to
put it bluntly, it sucks.  I really would like some advice on what would
be a good starter bike for me.  I do know one thing however, I need to
make my first bike a good one, because buying a second any time soon is
out of the question.  I am specifically interested in racing bikes, (CBR
600 F2, GSX-R 750).  I know that this may sound kind of crazy
considering that I've never had a bike before, but I am responsible, a
fast learner, and in love.  Please give me any advice that you think
would help me in my search, including places to look or even specific
bikes that you want to sell me.

    Thanks  :-)

    Jamie Belliveau (jbc9@andrew.cmu.edu)  



Rank 2: (Distance: 1.03924